In [0]:
# =============================================================================
# NOTEBOOK  : silver_purchase_orders
# PURPOSE   : bronze.purchase_orders → silver.purchase_orders
# LAYER     : Silver (clean, flatten nested JSON, CDC merge)
# FREQUENCY : Daily + CDC
# PATTERN   : spark.read + audit watermark (production pattern)
#
# TRANSFORM:
#   - order_date / expected_delivery_date / actual_delivery_date : String → DateType
#   - unit_cost        : Double → Decimal(10,2)
#   - quality_rating   : Double → Decimal(4,2)
#   - delivery_notes   : nested JSON string → flattened into 4 columns:
#                        carrier, tracking_number, delivery_status, delivery_notes_text
#   - order_year_month : derived from order_date → partition column
#   - status_priority  : derived for dedup — higher = later in lifecycle
#
# MERGE & DEDUP LOGIC (SCD Type 1 — CDC):
#   Merge key : po_id
#   Dedup     : two-step
#               Step 1 — dropDuplicates on (po_id, status) removes exact dupes
#               Step 2 — window on po_id ordered by status_priority DESC
#                        keeps record furthest in lifecycle when same po_id
#                        has multiple rows with different statuses
#               Done BEFORE .select() so status_priority available
#
#   Case 1 : po_id NOT in silver                  → INSERT
#   Case 2 : po_id IN silver, status changed      → UPDATE status only
#             status lifecycle: pending → shipped → delivered / cancelled
#             all other columns immutable once PO is created
#   Case 3 : po_id IN silver, no status change    → IGNORE
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit, get_last_successful_run_time
from utils.schema_registry import SILVER_PURCHASE_ORDERS_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_date, get_json_object,
    date_format, when, row_number
)
from pyspark.sql.types import DecimalType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["bronze_purchase_orders"]
TARGET_TABLE = cfg["tables"]["silver_purchase_orders"]
PIPELINE     = "silver_purchase_orders"

In [0]:
# ── Read + Transform + Merge ─────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
try:
    # ── INCREMENTAL READ ──────────────────────────────────────────────────────
    last_run_time = get_last_successful_run_time(spark, PROJECT_ROOT, PIPELINE)
 
    if last_run_time:
        bronze_df = (
            spark.read.table(SOURCE_TABLE)
            .filter(col("ingested_at") > lit(last_run_time))
        )
    else:
        bronze_df = spark.read.table(SOURCE_TABLE)
 
    rows_read = bronze_df.count()
    print(f"[READ] {rows_read} new bronze rows since last run")
 
    if rows_read == 0:
        print(f"[SKIP] No new rows to process")
        end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
                  rows_read=0, rows_written=0,
                  extra={"rows_inserted": "0", "rows_updated": "0"})
        raise SystemExit(0)
 
    # ── TRANSFORM ─────────────────────────────────────────────────────────────
    df = (
        bronze_df
 
        # 1. Date columns: String → DateType
        .withColumn("order_date",
            to_date(col("order_date")))
        .withColumn("expected_delivery_date",
            to_date(col("expected_delivery_date")))
        .withColumn("actual_delivery_date",
            to_date(col("actual_delivery_date")))
 
        # 2. Numeric precision
        .withColumn("unit_cost",
            col("unit_cost").cast(DecimalType(10, 2)))
        .withColumn("quality_rating",
            col("quality_rating").cast(DecimalType(4, 2)))
 
        # 3. Flatten delivery_notes nested JSON string
        #    Source: {"carrier":"UPS","tracking_number":"TRACK831632",
        #             "delivery_status":"delayed","notes":"Address issue"}
        .withColumn("carrier",
            get_json_object(col("delivery_notes"), "$.carrier"))
        .withColumn("tracking_number",
            get_json_object(col("delivery_notes"), "$.tracking_number"))
        .withColumn("delivery_status",
            get_json_object(col("delivery_notes"), "$.delivery_status"))
        .withColumn("delivery_notes_text",
            get_json_object(col("delivery_notes"), "$.notes"))
 
        # 4. Partition column derived from order_date
        .withColumn("order_year_month",
            date_format(col("order_date"), "yyyy-MM"))
 
        # 5. Status lifecycle priority — used for dedup only, dropped before merge
        #    Higher number = later in lifecycle = more recent state
        .withColumn("status_priority",
            when(col("status") == "pending",    lit(1))
            .when(col("status") == "shipped",   lit(2))
            .when(col("status") == "delivered", lit(3))
            .when(col("status") == "cancelled", lit(3))
            .otherwise(lit(0)))
 
        # 6. Silver audit columns
        .withColumn("processed_at",    current_timestamp())
        .withColumn("pipeline_run_id", lit(run_id))
        .withColumn("source_system",   lit("purchase_orders_csv"))
        # ingested_at carried from bronze automatically via .select()
    )
 
    # ── DEDUP ─────────────────────────────────────────────────────────────────
    # Step 1: remove exact duplicates (same po_id + same status)
    df = df.dropDuplicates(["po_id", "status"])
 
    # Step 2: if same po_id still has multiple rows with different statuses
    #         keep the one furthest in lifecycle (highest status_priority)
    #         Done BEFORE .select() so status_priority available
    window = (
        Window
        .partitionBy("po_id")
        .orderBy(col("status_priority").desc())
    )
 
    df = (
        df
        .withColumn("_row_num", row_number().over(window))
        .filter(col("_row_num") == 1)
        .drop("_row_num", "status_priority")
    )
 
    rows_after_dedup = df.count()
    if rows_read != rows_after_dedup:
        print(f"[DEDUP] dropped {rows_read - rows_after_dedup} duplicate PO rows")
 
    # Enforce silver schema — done AFTER dedup so status_priority available
    # Drops bronze-only columns (delivery_notes raw, source_file, ingested_date)
    df = df.select([f.name for f in SILVER_PURCHASE_ORDERS_SCHEMA.fields])
 
    # ── MERGE INTO SILVER (SCD Type 1 — CDC) ─────────────────────────────────
    # Case 2: po_id matched, status changed → UPDATE status only
    #         all other columns immutable once PO is created
    # Case 1: po_id not matched → INSERT
    # Case 3: po_id matched, no change → no rule fires → IGNORE
    (
        DeltaTable.forName(spark, TARGET_TABLE).alias("t")
        .merge(
            df.alias("s"),
            "t.po_id = s.po_id"
        )
        .whenMatchedUpdate(
            condition="t.status != s.status",
            set={
                "status":          "s.status",
                "processed_at":    "current_timestamp()",
                "pipeline_run_id": f"'{run_id}'"
                # all other columns immutable once PO is created
                # ingested_at NOT updated — preserve original bronze landing time
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
 
    metrics = (
        spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
        .select("operationMetrics")
        .collect()[0][0]
    )
    rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
    rows_updated  = int(metrics.get("numTargetRowsUpdated", 0))
 
    print(f"[DONE] inserted={rows_inserted} | updated={rows_updated}")
 
    end_audit(
        spark, PROJECT_ROOT, run_id, "SUCCESS",
        rows_read=rows_read,
        rows_written=rows_inserted + rows_updated,
        extra={
            "rows_inserted": str(rows_inserted),
            "rows_updated":  str(rows_updated)
        }
    )
 
except SystemExit:
    pass
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── CELL 3: Verify last run ───────────────────────────────────────────────────
from pyspark.sql.functions import col
 
# 1. Audit status
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()) \
    .limit(1) \
    .select("status", "rows_read", "rows_written", "extra_metadata") \
    .show(truncate=False)
 
# 2. Silver row count
print("Silver row count:", spark.read.table(TARGET_TABLE).count())
 
# 3. Nulls in key columns
print("Null po_ids:",
    spark.read.table(TARGET_TABLE)
    .filter(col("po_id").isNull())
    .count()
)
 
# 4. Status distribution
print("Status distribution:")
(
    spark.read.table(TARGET_TABLE)
    .groupBy("status")
    .count()
    .orderBy("status")
    .show(truncate=False)
)
 
# 5. Delta history
spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1") \
    .select("operation", "operationMetrics") \
    .show(truncate=False)